In [ ]:
# MIT License

# Copyright (c) 2024 Multifidelity Roofline Analysis

# Permission is hereby granted, free of charge, to any person obtaining a copy
# of this software and associated documentation files (the "Software"), to deal
# in the Software without restriction, including without limitation the rights
# to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
# copies of the Software, and to permit persons to whom the Software is
# furnished to do so, subject to the following conditions:

# The above copyright notice and this permission notice shall be included in all
# copies or substantial portions of the Software.

# THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
# IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
# FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
# AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
# LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
# OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
# SOFTWARE.

from GenZ import analyse_model, get_model_df, System
from GenZ.Models.get_language_model import einsum_test
import numpy as np

In [ ]:
%load_ext autoreload
%autoreload 2   

In [ ]:
class Einsum():
    def __init__(self, equation, dims):
        """
        equation: Einstein summation notation string
        dims: Dictionary of tensor dimensions keyed by the corresponding label in the equation
        """
        self.equation = equation
        self.dims = dims
    def get_size(self, tensor):
        return np.prod(tensor)
    def get_tensors(self):
        input_dims = self.equation.split('->')[0]
        input_a = [self.dims[label] for label in input_dims.split(',')[0]]
        input_b = [self.dims[label] for label in input_dims.split(',')[1]]
        output = [self.dims[label] for label in self.equation.split('->')[1]]
        return input_a, input_b, output

    def get_num_ops(self):
        """
        Compute the number of operations needed for the given einsum configuration.
        """
        input_dims = self.equation.split('->')[0]
        dim_labels = set(''.join(input_dims.split(',')))

        # The number of operations is the product of the dimensions involved in the contraction
        num_ops = np.prod([self.dims[label] for label in dim_labels])
        return num_ops

In [ ]:
op = Einsum('bhqd,hql->bhdl', {'b': 32, 'h': 64, 'q': 128, 'd': 256, 'l': 512})

In [ ]:
op.get_num_ops()

In [ ]:
op.get_tensors()

In [ ]:
list(map(op.get_size, [op.get_tensors()[0], op.get_tensors()[1], op.get_tensors()[2]]))

In [ ]:
model = einsum_test(equation='bld,dhq->blhq', einsum_vars={'b': 'b', 'l': 512, 'd': 2048, 'h': 12, 'q': 128})

In [ ]:
model

In [ ]:
get_model_df(model, system=System(), batch_size=1)